# End-to-End Hierarchical Cell Type Classification with HCE Loss (Option A)

Trains a C2S encoder + classification head end-to-end using **Hierarchical Cross-Entropy (HCE) loss** on the balanced lung dataset.

## Key approach
- C2S encoder + learnable classification head
- End-to-end training with **weighted** HCE loss (paper Eq. 7)
- Balanced dataset (capped max cells per type)
- **Option A — mixed-granularity labels**: each cell's label is randomly sampled from one of its valid annotation levels (`ann_level_1` → `ann_finest_level`) at every training step, so coarser classes appear as real targets and HCE genuinely activates.

## HCE Loss (paper Eqs. 4 & 7)
```
s  = softmax(logits) @ R.T       # adjusted scores (Eq. 4)
L  = -w_t * log(s_t + ε)         # weighted HCE loss (Eq. 7)
w_i = N / (C * n_i)              # scikit-learn balanced weights
```
Reachability matrix **R** has `R[parent, child] = 1` (parent→child semantics).

## Key corrections over the original notebook
- ✅ Mixed-granularity labels — HCE now provides genuine benefit over CE
- ✅ Class space = all observed labels across all annotation levels (not leaves + separate ancestors)
- ✅ Class weights `w_i = N / (C * n_i)` added to HCE loss (paper Eq. 7)
- ✅ Last non-padding token pooling (correct for decoder-only/causal LM)
- ✅ Reachability matrix semantics: PARENT→CHILD with transitive closure
- ✅ Best checkpoint saving by validation loss

## 1. Import Required Libraries

In [1]:
import os
import sys
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Add cell2sentence to path
sys.path.insert(0, '/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/src')

from cell2sentence.hce_trainer import build_reachability_matrix_from_ontology

print("✅ Libraries imported successfully!")
print("test")

/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/c2s-justin/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries imported successfully!
test


## 2. Configuration

In [ ]:
# File paths
LUNG_H5AD_PATH = '/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung.h5ad'
OUT_DIR = '/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung_hce_end_to_end_results_corrected'
BEST_MODEL_PATH = os.path.join(OUT_DIR, 'best_model.pt')
os.makedirs(OUT_DIR, exist_ok=True)

# Model configuration
C2S_MODEL_NAME = 'vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation'
TOP_K_GENES = 100

# Dataset balancing parameters
MAX_CELLS_PER_TYPE = 1000
MIN_CELLS_PER_TYPE = 20
ANN_COL = 'ann_finest_level'

# Training parameters
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
N_EPOCHS = 10
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01

# Evaluation split
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Device configuration
def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        device = torch.device("mps")
        print("Using Apple Metal Performance Shaders (MPS)")
    else:
        device = torch.device("cpu")
        print("Using CPU (no GPU detected)")
    return device

device = get_device()

print("\n" + "="*80)
print("CONFIGURATION SUMMARY")
print("="*80)
print(f"Dataset: {LUNG_H5AD_PATH}")
print(f"Output: {OUT_DIR}")
print(f"Best model path: {BEST_MODEL_PATH}")
print(f"Model: {C2S_MODEL_NAME.split('/')[-1]}")
print(f"Device: {device}")
print(f"\nDataset Balancing:")
print(f"  Max cells per type: {MAX_CELLS_PER_TYPE}")
print(f"  Min cells per type: {MIN_CELLS_PER_TYPE}")
print(f"\nTraining Config:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {N_EPOCHS}")
print(f"  Top K genes: {TOP_K_GENES}")

## 3. Load and Balance Dataset

In [ ]:
print("\n" + "="*80)
print("LOADING AND BALANCING DATASET")
print("="*80)

# Load dataset
print(f"\n📚 Loading {LUNG_H5AD_PATH}...")
adata = sc.read_h5ad(LUNG_H5AD_PATH, backed='r')
print(f"   Original shape: {adata.shape}")

# Check for annotation column
if ANN_COL not in adata.obs.columns:
    ann_cols = [col for col in adata.obs.columns if 'cell' in col.lower() or 'type' in col.lower()]
    if ann_cols:
        ANN_COL = ann_cols[0]
        print(f"⚠️  '{ANN_COL}' not found. Using '{ANN_COL}' instead")
    else:
        raise ValueError(f"Could not find annotation column")

# Show original distribution
print(f"\n📊 Original class distribution (top 10):")
orig_counts = adata.obs[ANN_COL].value_counts().sort_values(ascending=False)
for idx, (cell_type, count) in enumerate(orig_counts.head(10).items(), 1):
    pct = count / len(adata.obs) * 100
    print(f"   {idx:2d}. {cell_type:40s}: {count:5d} cells ({pct:5.1f}%)")
if len(orig_counts) > 10:
    print(f"   ... and {len(orig_counts) - 10} more cell types")

# Balance the dataset
print(f"\n🎯 Balancing dataset (capping at {MAX_CELLS_PER_TYPE} per type)...")
labels = adata.obs[ANN_COL].astype(str)
counts = labels.value_counts()

keep_labels = counts[counts >= MIN_CELLS_PER_TYPE].index.tolist()
print(f"   Keeping {len(keep_labels)} classes with >= {MIN_CELLS_PER_TYPE} cells")
print(f"   Dropping {len(counts) - len(keep_labels)} classes (too rare)")

# Sample up to MAX_CELLS_PER_TYPE from each label
sampled_idx = []
for lbl in keep_labels:
    idxs = np.where(labels.values == lbl)[0]
    n_keep = min(len(idxs), MAX_CELLS_PER_TYPE)
    chosen = np.random.choice(idxs, size=n_keep, replace=False)
    sampled_idx.extend(chosen.tolist())

# Create balanced AnnData
sampled_idx = np.array(sampled_idx, dtype=int)
sampled_idx = np.sort(sampled_idx)
adata_balanced = adata[sampled_idx].to_memory()

# Remove unknown/invalid annotations
print(f"\n🧹 Removing unknown/invalid annotations...")
unknown_terms = ['unknown', 'Unknown', 'UNKNOWN', 'nan', 'NA', 'N/A', 'none', 'None']
valid_mask = ~adata_balanced.obs[ANN_COL].isin(unknown_terms)
print(f"   Found {(~valid_mask).sum()} unknown cells")
adata_balanced = adata_balanced[valid_mask].copy()

print(f"\n✅ Balanced dataset created (after removing unknowns):\"")
print(f"   Total cells: {adata_balanced.n_obs}")
print(f"   Cell types: {len(adata_balanced.obs[ANN_COL].unique())}\"")

# Show new distribution
new_counts = adata_balanced.obs[ANN_COL].value_counts().sort_values(ascending=False)
print(f"\n📊 Balanced class distribution (top 10):\"")
for idx, (cell_type, count) in enumerate(new_counts.head(10).items(), 1):
    pct = count / len(adata_balanced.obs) * 100
    print(f"   {idx:2d}. {cell_type:40s}: {count:5d} cells ({pct:5.1f}%)")
if len(new_counts) > 10:
    print(f"   ... and {len(new_counts) - 10} more cell types")

# Save counts
orig_counts.to_csv(os.path.join(OUT_DIR, 'original_counts.csv'), header=['count'])
new_counts.to_csv(os.path.join(OUT_DIR, 'balanced_counts.csv'), header=['count'])
print(f"\n✅ Saved dataset counts")

In [ ]:

# Verify minority cell types are represented
print(f"\n📋 VERIFYING MINORITY CELL TYPE REPRESENTATION:")
print(f"\n🔍 Checking for Alveolar macrophage family members...")

alveolar_family = [
    'Alveolar macrophages',
    'Alveolar Mph MT-positive',
    'Alveolar Mph CCL3+',
    'Alveolar Mph proliferating'
]

for cell_type in alveolar_family:
    if cell_type in adata_balanced.obs[ANN_COL].values:
        count = (adata_balanced.obs[ANN_COL] == cell_type).sum()
        pct = (count / len(adata_balanced.obs)) * 100
        print(f"   ✅ {cell_type:<40s}: {count:>6,} cells ({pct:>5.2f}%)")
    else:
        print(f"   ❌ {cell_type:<40s}: NOT FOUND")

# Check all cell types present
all_present_cells = adata_balanced.obs[ANN_COL].unique()
print(f"\n✅ Total cell types in balanced dataset: {len(all_present_cells)}")
print(f"   All Alveolar family members present: {all(ct in all_present_cells for ct in alveolar_family)}")


## 4. Build Cell Type Hierarchy

In [ ]:
print("\n" + "="*80)
print("BUILDING CELL TYPE HIERARCHY (WITH PROPER STOPPING)")
print("="*80)

# Define validation function
def is_valid_annotation(value):
    """Check if annotation is a valid cell type (not missing/unknown)."""
    if pd.isna(value):
        return False
    value_str = str(value).strip()
    invalid_terms = ['unknown', 'none', 'na', 'n/a', 'nan', '']
    return value_str.lower() not in invalid_terms

# Find annotation level columns (HLCA standard)
level_columns = sorted([col for col in adata_balanced.obs.columns 
                        if col.startswith('ann_level_') and col[-1].isdigit()])

print(f"\n🔍 Found {len(level_columns)} hierarchical annotation levels:")
for col in level_columns:
    n_unique = adata_balanced.obs[col].nunique()
    n_not_na = adata_balanced.obs[col].notna().sum()
    print(f"  - {col}: {n_unique} unique values ({n_not_na}/{len(adata_balanced.obs)} cells annotated)")

# Verify annotation hierarchy structure
print(f"\n📋 Verifying hierarchy structure (sample cell annotations):")
sample_idx = 0
for col in level_columns:
    val = adata_balanced.obs.iloc[sample_idx][col]
    print(f"  {col}: {val}")
print(f"  {ANN_COL}: {adata_balanced.obs.iloc[sample_idx][ANN_COL]}")

if len(level_columns) < 1:
    print("\n⚠️  WARNING: Could not find hierarchical annotation levels.")
    ontology_dict = {ct: None for ct in adata_balanced.obs[ANN_COL].unique()}
    all_cell_types = list(adata_balanced.obs[ANN_COL].unique())
else:
    # Build hierarchy from annotation levels WITH PROPER STOPPING LOGIC
    print(f"\n🏗️  Building hierarchy from {len(level_columns)} annotation levels + leaf column (STOPPING at Unknown/None)...")
    
    ontology_dict = {}
    all_cell_types = set()
    
    # Build ontology with STOPPING logic
    for idx in range(len(adata_balanced.obs)):
        # Extract valid annotations only (STOP at first None/Unknown)
        valid_path = []
        for level_col in level_columns:
            value = adata_balanced.obs.iloc[idx][level_col]
            if is_valid_annotation(value):
                valid_path.append((level_col, value))
            else:
                # STOP - don't continue to deeper levels
                break
        
        # Always append the leaf (ANN_COL) if valid
        leaf_value = adata_balanced.obs.iloc[idx][ANN_COL]
        if is_valid_annotation(leaf_value):
            valid_path.append((ANN_COL, leaf_value))
        
        # Add all valid cell types
        for _, cell_type in valid_path:
            all_cell_types.add(cell_type)
        
        # Build parent-child relationships
        for i in range(len(valid_path) - 1):
            child = valid_path[i + 1][1]
            parent = valid_path[i][1]
            
            # Only add if not already in ontology (keep first occurrence)
            if child not in ontology_dict:
                ontology_dict[child] = parent
    
    # Add root nodes (level 1 annotations)
    for idx in range(len(adata_balanced.obs)):
        if is_valid_annotation(adata_balanced.obs.iloc[idx]['ann_level_1']):
            root = adata_balanced.obs.iloc[idx]['ann_level_1']
            if root not in ontology_dict:
                ontology_dict[root] = None
    
    print(f"\n✅ Built hierarchy with PROPER STOPPING LOGIC:")
    print(f"   Total cell types: {len(all_cell_types)}")
    print(f"   Parent-child relationships: {len([v for v in ontology_dict.values() if v is not None])}")
    print(f"   Root nodes: {len([v for v in ontology_dict.values() if v is None])}")
    
    # Show example paths
    print(f"\n📊 Example parent-child relationships:")
    example_count = 0
    for child, parent in sorted(ontology_dict.items(), key=lambda x: (x[1] is None, x[0])):
        if parent is not None and example_count < 10:
            print(f"   {child} → {parent}")
            example_count += 1
    
    # Show root nodes
    roots = [child for child, parent in ontology_dict.items() if parent is None]
    print(f"\n📦 Root nodes ({len(roots)}):")
    for root in sorted(roots):
        print(f"   • {root}")
    
    print(f"\n✅ KEY IMPROVEMENT: Hierarchy now stops at Unknown/None and always includes the leaf column {ANN_COL}")
    print(f"   This prevents 'Unknown'/'None' from being treated as cell type names")
    print(f"   allowing proper parent-child relationships to be built, including leaves.")

# Save ontology
ontology_df = pd.DataFrame(list(ontology_dict.items()), columns=['child', 'parent'])
ontology_df.to_csv(os.path.join(OUT_DIR, 'ontology.csv'), index=False)
print(f"\n✅ Saved ontology to {os.path.join(OUT_DIR, 'ontology.csv')}")

## 5. Prepare Cell Text Data

In [ ]:
print("\n" + "="*80)
print("PREPARING CELL TEXT DATA")
print("="*80)

# Helper functions
def cell_to_text(cell_vector, gene_names, top_k=100):
    """Convert cell expression to top-k gene names."""
    if hasattr(cell_vector, "toarray"):
        vec = cell_vector.toarray().flatten()
    else:
        vec = np.asarray(cell_vector).flatten()
    if vec.size == 0:
        return ""
    top_idx = np.argsort(vec)[-top_k:][::-1]
    top_genes = [str(gene_names[i]) for i in top_idx if gene_names[i]]
    return " ".join(top_genes)

# Convert cells to text
print(f"\n📝 Converting {adata_balanced.n_obs} cells to text...")
cell_texts = []
for i in range(adata_balanced.n_obs):
    text = cell_to_text(adata_balanced.X[i], adata_balanced.var_names, top_k=TOP_K_GENES)
    cell_texts.append(text)

# Filter empty
keep_idx = [i for i, t in enumerate(cell_texts) if isinstance(t, str) and t.strip() != ""]
cell_texts  = [cell_texts[i] for i in keep_idx]
cell_ids    = adata_balanced.obs_names[keep_idx].tolist()
cell_labels = adata_balanced.obs[ANN_COL].iloc[keep_idx].astype(str).values

print(f"✅ Converted {len(cell_texts)} cells to text")
print(f"   Sample: {cell_texts[0][:100]}...")

# ── Option A: collect all valid annotation levels per cell ──────────────────
# Walk ann_level_1 → ... → ann_finest_level, stopping at the first
# invalid (None/Unknown) entry. Each cell gets a list of labels from
# coarsest to finest that will be randomly sampled during training.
# This is the key change that makes HCE work: coarser labels become
# real training targets so the reachability matrix actively shapes gradients.
print(f"\n📊 [Option A] Collecting multi-granularity labels per cell...")

level_columns_all = level_columns + [ANN_COL]   # ordered coarser → finest
cell_all_labels = []   # list[list[str]], one per kept cell

for orig_i in keep_idx:
    valid_labels = []
    for col in level_columns_all:
        val = adata_balanced.obs.iloc[orig_i][col]
        if is_valid_annotation(val):
            valid_labels.append(str(val))
        else:
            break   # stop at first invalid level (mirrors hierarchy-build logic)
    if not valid_labels:
        valid_labels = [str(adata_balanced.obs.iloc[orig_i][ANN_COL])]
    cell_all_labels.append(valid_labels)

n_levels_avg = np.mean([len(ls) for ls in cell_all_labels])
unique_across_levels = set(l for ls in cell_all_labels for l in ls)
print(f"✅ Multi-granularity labels collected:")
print(f"   Average annotation levels per cell : {n_levels_avg:.2f}")
print(f"   Unique labels across all levels     : {len(unique_across_levels)}")

## 6. Create Dataset and Model

In [ ]:
# Load tokenizer
print(f"\n🔄 Loading Cell2Sentence tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(C2S_MODEL_NAME)
print(f"✅ Tokenizer loaded")

# ── [Option A] Build class vocabulary from ALL observed labels ───────────────
# Every annotation string that appears in training (at any granularity level)
# is a potential training target. This is what makes HCE meaningful: coarser
# labels appear as targets so the reachability matrix actively shapes gradients.
print(f"\n🏗️  [Option A] Building class vocabulary from all annotation levels...")

all_observed_labels = set(l for ls in cell_all_labels for l in ls)
class_names  = sorted(all_observed_labels)
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

# Leaf classes = finest-level labels (used exclusively for evaluation)
leaf_classes = sorted({l for l in adata_balanced.obs[ANN_COL].unique() if l in class_to_idx})
leaf_to_idx  = {leaf: class_to_idx[leaf] for leaf in leaf_classes}
leaf_indices = [class_to_idx[leaf] for leaf in leaf_classes]

print(f"\n✅ Class vocabulary built:")
print(f"   Total classes (all levels) : {len(class_names)}")
print(f"   Finest-level (leaf) classes: {len(leaf_classes)}")
print(f"   Coarser-level classes      : {len(class_names) - len(leaf_classes)}")

# Encode all-level labels per cell as index lists
cell_all_labels_encoded = [
    [class_to_idx[l] for l in ls if l in class_to_idx]
    for ls in cell_all_labels
]

# Finest-level encoding — for stratified splitting and evaluation only
labels_encoded = np.array(
    [class_to_idx[cell_labels[i]] for i in range(len(cell_labels))], dtype=int
)
print(f"\n✅ Label encoding complete: {len(labels_encoded)} cells, {len(class_names)} classes")

# ── [Option A] Dataset: randomly samples annotation level during training ────
class CellTextDataset(Dataset):
    """During training randomly samples one annotation level per cell.
    During eval always uses the finest-level (last) label."""
    def __init__(self, texts, all_labels_encoded, tokenizer, max_length=512, is_train=True):
        self.texts              = texts
        self.all_labels_encoded = all_labels_encoded   # list[list[int]]
        self.tokenizer          = tokenizer
        self.max_length         = max_length
        self.is_train           = is_train

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text   = self.texts[idx]
        labels = self.all_labels_encoded[idx]

        if self.is_train:
            label = labels[np.random.randint(len(labels))]  # random level
        else:
            label = labels[-1]                               # finest level only

        encoding = self.tokenizer(
            text, padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":         torch.tensor(label, dtype=torch.long),
        }

# Stratified split by finest-level label
idx = np.arange(len(labels_encoded))
idx_trainval, idx_test, y_trainval, y_test = train_test_split(
    idx, labels_encoded, test_size=TEST_FRAC, stratify=labels_encoded, random_state=RANDOM_SEED
)
relative_val_frac = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)
idx_train, idx_val, y_train, y_val = train_test_split(
    idx_trainval, y_trainval, test_size=relative_val_frac, stratify=y_trainval, random_state=RANDOM_SEED
)

print(f"\n✅ Data split: Train={len(idx_train)}, Val={len(idx_val)}, Test={len(idx_test)}")

train_texts = [cell_texts[i] for i in idx_train]
val_texts   = [cell_texts[i] for i in idx_val]
test_texts  = [cell_texts[i] for i in idx_test]

train_all_labels = [cell_all_labels_encoded[i] for i in idx_train]
val_all_labels   = [cell_all_labels_encoded[i] for i in idx_val]
test_all_labels  = [cell_all_labels_encoded[i] for i in idx_test]

train_dataset = CellTextDataset(train_texts, train_all_labels, tokenizer, is_train=True)
val_dataset   = CellTextDataset(val_texts,   val_all_labels,   tokenizer, is_train=False)
test_dataset  = CellTextDataset(test_texts,  test_all_labels,  tokenizer, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f"✅ DataLoaders created (train samples randomly across {n_levels_avg:.1f} levels/cell)")

## 7. Define Model Architecture

In [ ]:
# Load C2S encoder
print(f"\n🔄 Loading C2S encoder...")
c2s_model = AutoModel.from_pretrained(C2S_MODEL_NAME)
print(f"✅ C2S encoder loaded")

hidden_size = c2s_model.config.hidden_size
print(f"   Hidden size: {hidden_size}")

class ClassificationHead(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.1):
        super().__init__()
        self.dropout    = nn.Dropout(dropout)
        self.dense      = nn.Linear(input_dim, 256)
        self.dense2     = nn.Linear(256, num_classes)
        self.activation = nn.ReLU()

    def forward(self, hidden_states):
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.dense(hidden_states)
        hidden_states = self.activation(hidden_states)
        hidden_states = self.dropout(hidden_states)
        return self.dense2(hidden_states)

class C2SClassifier(nn.Module):
    def __init__(self, encoder, num_classes, hidden_size):
        super().__init__()
        self.encoder    = encoder
        self.classifier = ClassificationHead(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs     = self.encoder(input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state           # (B, T, H)

        # [Fix] Last non-padding token pooling.
        # C2S uses Pythia (GPT/decoder-only causal LM): the last real token
        # has attended to the full sequence and is the correct representation.
        # Mean pooling (original) averages partial-context prefix positions.
        seq_lengths = attention_mask.sum(dim=1) - 1       # (B,) 0-indexed
        batch_size  = last_hidden.size(0)
        last_token  = last_hidden[
            torch.arange(batch_size, device=last_hidden.device), seq_lengths
        ]                                                  # (B, H)

        return self.classifier(last_token)

model = C2SClassifier(c2s_model, len(class_names), hidden_size)
model.to(device)
print(f"\n✅ Model created and moved to {device}")
print(f"   Output classes : {len(class_names)} (all annotation levels)")
print(f"   Pooling        : last non-padding token (correct for causal LM)")

## 8. Build Reachability Matrix and HCE Loss (CORRECTED)

In [ ]:
# Build reachability matrix over ALL class_names (all annotation levels)
print("\n" + "="*80)
print("BUILDING REACHABILITY MATRIX FOR HCE LOSS")
print("="*80)

def build_reachability_matrix_hce(ontology_dict, class_names, device):
    """
    Build reachability matrix with PARENT -> CHILD semantics.
    R[i, j] = 1 if class j is a descendant of class i (or j == i).
    Adjusted scores: s = probs @ R.T  (paper Eq. 4)
    """
    n   = len(class_names)
    c2i = {name: idx for idx, name in enumerate(class_names)}
    R   = np.eye(n, dtype=np.float32)

    skipped = 0
    for child, parent in ontology_dict.items():
        if parent is None:
            continue
        if child in c2i and parent in c2i:
            R[c2i[parent], c2i[child]] = 1.0
        else:
            skipped += 1
    if skipped:
        print(f"⚠️  Skipped {skipped} ontology edges not present in class_names")

    print("📊 Computing transitive closure (Floyd-Warshall)...")
    for k in range(n):
        for i in range(n):
            if R[i, k] == 0:
                continue
            for j in range(n):
                if R[k, j]:
                    R[i, j] = 1.0

    return torch.tensor(R, dtype=torch.float32, device=device)

reachability_matrix = build_reachability_matrix_hce(ontology_dict, class_names, device)
nz = torch.count_nonzero(reachability_matrix).item()
print(f"\n✅ Reachability matrix: {tuple(reachability_matrix.shape)}")
print(f"   Non-zero (incl. diagonal): {nz}")
print(f"   Off-diagonal relations   : {nz - len(class_names)}")

# Index helpers
leaf_index_set   = set(leaf_indices)
ancestor_indices = [i for i in range(len(class_names)) if i not in leaf_index_set]

# ── [Fix] Class weights w_i = N / (C * n_i)  (paper Eq. 7) ─────────────────
print("\n📊 Computing class weights (paper Eq. 7: w_i = N / (C * n_i))...")
from collections import Counter
train_finest_counts = Counter(y_train.tolist())
N_train    = len(y_train)
C_observed = len(train_finest_counts)

class_weights = torch.zeros(len(class_names), dtype=torch.float32, device=device)

# Leaf weights from direct counts
for label_idx, count in train_finest_counts.items():
    class_weights[label_idx] = N_train / (C_observed * count)

# Coarser-level weights: effective count = sum of descendant leaf counts
for anc_idx in ancestor_indices:
    eff_count = sum(
        train_finest_counts.get(j, 0)
        for j in leaf_indices
        if reachability_matrix[anc_idx, j] > 0
    )
    if eff_count > 0:
        class_weights[anc_idx] = N_train / (C_observed * eff_count)

non_zero_w = (class_weights > 0).sum().item()
print(f"✅ Class weights: {non_zero_w}/{len(class_names)} non-zero, "
      f"range [{class_weights[class_weights>0].min():.4f}, {class_weights.max():.4f}]")

# ── [Fix] Weighted HCE Loss (paper Eq. 7) ────────────────────────────────────
class HCELoss(nn.Module):
    """
    Hierarchical Cross-Entropy Loss with class weights (paper Eq. 7).
    R[parent, child] = 1  (PARENT->CHILD)
    s = softmax(logits) @ R.T          (paper Eq. 4)
    L = -w_t * log(s_t + eps)          (paper Eq. 7)
    """
    def __init__(self, reachability_matrix, class_weights=None, epsilon=1e-8):
        super().__init__()
        self.register_buffer("reachability_matrix", reachability_matrix)
        self.epsilon = epsilon
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights)
        else:
            self.class_weights = None

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)                        # [B, C]
        s     = probs @ self.reachability_matrix.T                  # [B, C]
        s     = torch.clamp(s, min=self.epsilon)
        log_s = torch.log(s)                                        # [B, C]

        bidx  = torch.arange(len(targets), device=targets.device)
        log_st = log_s[bidx, targets]                               # [B]

        if self.class_weights is not None:
            w    = self.class_weights[targets]                      # [B]
            loss = -(w * log_st).mean()
        else:
            loss = -log_st.mean()
        return loss

hce_criterion = HCELoss(reachability_matrix, class_weights=class_weights)
criterion     = hce_criterion   # alias

print("\n✅ HCE loss initialized (weighted, paper Eq. 7)")
print("   s = softmax(logits) @ R.T")
print("   L = -w_t * log(s_t + ε)")

In [ ]:
print("\n" + "="*80)
print("VERIFICATION: HCE MATRIX SEMANTICS AND OPTION A ACTIVATION")
print("="*80)

n = reachability_matrix.shape[0]
diag_ok = torch.allclose(
    torch.diag(reachability_matrix),
    torch.ones(n, device=reachability_matrix.device)
)
print(f"Identity on diagonal       : {diag_ok}")
print(f"Matrix shape               : {tuple(reachability_matrix.shape)}")

valid_edges, checked = 0, 0
for child, parent in ontology_dict.items():
    if parent is None or child not in class_to_idx or parent not in class_to_idx:
        continue
    checked += 1
    p, c = class_to_idx[parent], class_to_idx[child]
    if reachability_matrix[p, c] > 0:
        valid_edges += 1
print(f"Direct edges (parent→child): {valid_edges}/{checked}")

# Confirm HCE is active: find a coarser class with descendant leaves
coarser_example = None
for anc_idx in ancestor_indices:
    desc = [j for j in leaf_indices if reachability_matrix[anc_idx, j] > 0]
    if desc:
        coarser_example = (class_names[anc_idx], [class_names[j] for j in desc[:3]])
        break

if coarser_example:
    anc_name, children = coarser_example
    print(f"\nExample coarser training target: '{anc_name}'")
    print(f"  Descendant leaves (sample)    : {children}")
    print("  → When sampled as a label, HCE propagates gradient to all descendants ✅")

print(f"\n✅ Option A confirmed active:")
print(f"   {len(ancestor_indices)} coarser classes appear as training targets")
print(f"   Each training step randomly samples from {n_levels_avg:.1f} levels/cell on average")

## 9. Training Loop

In [ ]:
print("\n" + "="*80)
print("TRAINING — HCE LOSS WITH OPTION A (MIXED-GRANULARITY LABELS)")
print("="*80)

optimizer        = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps      = len(train_loader) * N_EPOCHS
warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_STEPS)
main_scheduler   = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_steps - WARMUP_STEPS))

# Project any ancestor prediction to its best descendant leaf at eval time
def project_to_leaf(logits_row):
    raw_idx = torch.argmax(logits_row).item()
    if raw_idx in leaf_index_set:
        return raw_idx
    desc = [li for li in leaf_indices if reachability_matrix[raw_idx, li] > 0]
    if desc:
        best = torch.argmax(logits_row[torch.tensor(desc, device=logits_row.device)]).item()
        return desc[best]
    return leaf_indices[torch.argmax(logits_row[torch.tensor(leaf_indices, device=logits_row.device)]).item()]

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch in tqdm(loader, desc="Training"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        optimizer.zero_grad()
        loss = criterion(model(input_ids, attention_mask), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    """Evaluation always uses finest-level labels (Dataset is_train=False)."""
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            logits = model(input_ids, attention_mask)
            total_loss += criterion(logits, labels).item()
            for row in logits:
                all_preds.append(project_to_leaf(row))
            all_labels.extend(labels.cpu().numpy().tolist())
    return total_loss / len(loader), accuracy_score(all_labels, all_preds), all_preds, all_labels

train_losses, val_losses, val_accs = [], [], []
best_val_loss = float("inf")
global_step   = 0

print(f"\n📊 Training for {N_EPOCHS} epochs...\n")

for epoch in range(N_EPOCHS):
    print(f"\nEpoch {epoch+1}/{N_EPOCHS}")
    train_loss = train_epoch(model, train_loader, hce_criterion, optimizer, device)
    train_losses.append(train_loss)
    global_step += len(train_loader)

    if global_step < WARMUP_STEPS:
        warmup_scheduler.step()
    else:
        main_scheduler.step()

    val_loss, val_acc, _, _ = evaluate(model, val_loader, hce_criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({"model_state_dict": model.state_dict()}, BEST_MODEL_PATH)
        print(f"  ✅ Best val loss {best_val_loss:.4f} — checkpoint saved")

    print(f"  Train Loss   : {train_loss:.4f}")
    print(f"  Val Loss     : {val_loss:.4f}")
    print(f"  Val Accuracy : {val_acc:.4f}")

print(f"\n✅ Training complete! Best val loss: {best_val_loss:.4f}")

## 10. Evaluation on Test Set

In [ ]:
print("\n" + "="*80)
print("TEST SET EVALUATION")
print("="*80)

# Load best checkpoint for evaluation
if os.path.exists(BEST_MODEL_PATH):
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Loaded best checkpoint from {BEST_MODEL_PATH}")
else:
    print(f"⚠️ Best checkpoint not found at {BEST_MODEL_PATH}; using last-epoch weights")

test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, hce_criterion, device)

print(f"\n✅ Test Results:")
print(f"   Test Loss: {test_loss:.4f}")
print(f"   Test Accuracy: {test_acc:.4f}")

# Create confusion matrix over leaf classes only
leaf_label_indices = leaf_indices
cm = confusion_matrix(test_labels, test_preds, labels=leaf_label_indices)
print(f"   Confusion Matrix (leaf-only): {cm.shape}")

# Per-class metrics
precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, labels=leaf_label_indices
)

per_class_metrics = pd.DataFrame({
    'cell_type': [class_names[i] for i in leaf_label_indices],
    'precision': precision,
    'recall': recall,
    'f1': f1
}).sort_values('f1', ascending=False)

metrics_path = os.path.join(OUT_DIR, 'per_class_metrics.csv')
per_class_metrics.to_csv(metrics_path, index=False)

print(f"\n📊 Top 15 Cell Types by F1 Score:")
print(per_class_metrics[['cell_type', 'precision', 'recall', 'f1']].head(15).to_string(index=False))

print(f"\n✅ Saved per-class metrics to {metrics_path}")

## 11. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax = axes[0]
epochs = np.arange(1, N_EPOCHS + 1)
ax.plot(epochs, train_losses, 'o-', label='Train Loss', linewidth=2, markersize=8)
ax.plot(epochs, val_losses, 's-', label='Val Loss', linewidth=2, markersize=8)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('HCE Loss During Training (CORRECTED)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Accuracy
ax = axes[1]
ax.plot(epochs, val_accs, 'o-', label='Val Accuracy', linewidth=2, markersize=8, color='green')
ax.axhline(y=test_acc, color='red', linestyle='--', label=f'Test Accuracy: {test_acc:.4f}', linewidth=2)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Validation & Test Accuracy', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim([0, 1.0])

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, 'training_curves.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
print(f"✅ Saved training curves to {fig_path}")
plt.show()

# Confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(22, 20))
im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', aspect='auto')
ax.set_title('Confusion Matrix - HCE Trained Model (CORRECTED)\n(Normalized by True Label)', 
             fontsize=8, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.colorbar(im, ax=ax, label='Normalized Count')

# Show all labels (no reduction)
tick_positions = np.arange(len(leaf_classes))
tick_labels = leaf_classes

ax.set_xticks(tick_positions)
ax.set_yticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=90, ha='center', fontsize=5)
ax.set_yticklabels(tick_labels, fontsize=5)

plt.tight_layout()
cm_path = os.path.join(OUT_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=200, bbox_inches='tight')
print(f"✅ Saved confusion matrix to {cm_path}")
plt.show()

# Per-class F1
fig, ax = plt.subplots(figsize=(12, 10))

top_n = 15
top_metrics = per_class_metrics.head(top_n)
x = np.arange(len(top_metrics))
width = 0.25

ax.barh(x - width, top_metrics['precision'].values, width, label='Precision', alpha=0.8)
ax.barh(x, top_metrics['recall'].values, width, label='Recall', alpha=0.8)
ax.barh(x + width, top_metrics['f1'].values, width, label='F1', alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(top_metrics['cell_type'].values, fontsize=10)
ax.set_xlabel('Score', fontsize=11)
ax.set_title(f'Top {top_n} Cell Types - Performance Metrics', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)
ax.set_xlim([0, 1.0])

plt.tight_layout()
f1_path = os.path.join(OUT_DIR, 'per_class_metrics.png')
plt.savefig(f1_path, dpi=200, bbox_inches='tight')
print(f"✅ Saved per-class metrics plot to {f1_path}")
plt.show()

## 12. Summary Report

In [ ]:
print("\n" + "="*80)
print("END-TO-END HCE TRAINING - SUMMARY REPORT (CORRECTED)")
print("="*80)

summary_text = f"""
DATASET CONFIGURATION
=====================
Original dataset:       {orig_counts.sum():,} cells
Balanced dataset:       {len(labels_encoded):,} cells
Max cells per type:     {MAX_CELLS_PER_TYPE}
Min cells per type:     {MIN_CELLS_PER_TYPE}
Leaf cell types:        {len(leaf_classes)}
Total logits classes:   {len(class_names)} (leaves + ancestors)

HIERARCHY INFORMATION
====================
Total unique nodes:     {len(class_names)}
Parent-child relations: {len([v for v in ontology_dict.values() if v is not None])}
Reachability matrix:    {reachability_matrix.shape} (with transitive closure)
Non-diagonal entries:   {torch.count_nonzero(reachability_matrix).item() - len(class_names)}

MODEL CONFIGURATION
===================
Encoder:                {C2S_MODEL_NAME.split('/')[-1]}
Classification head:    Dense(hidden_size -> 256) -> ReLU -> Dense(256 -> num_classes)
Loss function:          Hierarchical Cross-Entropy (HCE) - CORRECTED
Optimizer:              AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})
Epochs:                 {N_EPOCHS}
Batch size:             {BATCH_SIZE}

KEY CORRECTIONS IN THIS VERSION
================================
✅ Reachability matrix directionality: Parent→Child (was backwards)
✅ Transitive closure: Floyd-Warshall algorithm applied
✅ HCE loss formula: Uses matrix[:, true_label] for ancestor selection
✅ Logit space includes ancestors (allows ancestor credit during training)
✅ Leaf column appended to ontology build (no missing leaves)

TRAINING RESULTS
================
Final Train Loss:       {train_losses[-1]:.4f}
Final Val Loss:         {val_losses[-1]:.4f}
Final Val Accuracy:     {val_accs[-1]:.4f}

TEST SET PERFORMANCE
====================
Test Loss:              {test_loss:.4f}
Test Accuracy:          {test_acc:.4f}

Top 5 Cell Types by F1 Score:
"""

for idx, row in per_class_metrics.head(5).iterrows():
    summary_text += f"  {row['cell_type']:40s}: F1={row['f1']:.4f} (P={row['precision']:.4f}, R={row['recall']:.4f})\n"

summary_text += f"""
KEY ADVANTAGES OF CORRECTED HCE LOSS
====================================
• Hierarchy is correctly incorporated into the training objective
• All {torch.count_nonzero(reachability_matrix).item() - len(class_names)} hierarchical relationships are now active
• Ancestor predictions receive partial credit while training
• Transitive closure ensures multi-level hierarchy is properly represented
• End-to-end optimization of encoder + classifier

OUTPUT FILES
============
• training_curves.png - Loss and accuracy during training
• confusion_matrix.png - Normalized confusion matrix heatmap (leaf-only)
• per_class_metrics.png - Precision/Recall/F1 for top cell types
• per_class_metrics.csv - Complete per-class metrics
• ontology.csv - Cell type hierarchy
• balanced_counts.csv - Class distribution after balancing
"""

print(summary_text)

# Save summary
summary_path = os.path.join(OUT_DIR, "hce_training_summary.txt")
with open(summary_path, 'w') as f:
    f.write(summary_text)

print(f"\n✅ Summary saved to {summary_path}")
print(f"\n✅ All outputs saved to: {OUT_DIR}")

In [ ]:
print("\n" + "="*80)
print("WORST PERFORMING CELL TYPES ANALYSIS")
print("="*80)

# Get worst performing cell types
worst_metrics = per_class_metrics.sort_values('f1', ascending=True).head(10)

print(f"\n❌ WORST 10 CELL TYPES BY F1 SCORE:")
print(worst_metrics[['cell_type', 'precision', 'recall', 'f1']].to_string(index=False))

# Analyze hierarchy structure for worst performers
print(f"\n📊 HIERARCHY ANALYSIS FOR WORST PERFORMERS:")
print(f"{'Cell Type':<40} {'Parent':<40} {'F1 Score':<10} {'Issue':<30}")
print("-" * 120)

for idx, row in worst_metrics.iterrows():
    cell_type = row['cell_type']
    f1_score = row['f1']
    parent = ontology_dict.get(cell_type, 'NOT FOUND')
    
    # Identify issues
    issue = ""
    if f1_score < 0.7:
        issue = "VERY LOW F1"
    if parent is None:
        issue += " (ROOT NODE)" if issue else "(ROOT NODE)"
    if parent == cell_type or parent == 'nan':
        issue += " (SELF/INVALID PARENT)" if issue else "(SELF/INVALID PARENT)"
    
    print(f"{cell_type:<40} {str(parent):<40} {f1_score:<10.4f} {issue:<30}")

# Find cells with confused predictions
print(f"\n🔍 CONFUSION ANALYSIS:")
print(f"{'Cell Type':<40} {'Most Confused With':<40} {'Confusion %':<15}")
print("-" * 95)

for cell_idx, cell_type in enumerate(leaf_classes):
    cell_f1 = per_class_metrics[per_class_metrics['cell_type'] == cell_type]['f1'].values[0]
    
    # Only analyze worst performers
    if cell_f1 >= 0.85:
        continue
    
    # Get which class this was most confused with
    confusion_row = cm[cell_idx, :]
    total_incorrect = confusion_row.sum() - confusion_row[cell_idx]
    
    if total_incorrect > 0:
        # Find the most common misclassification
        misclass_idx = np.argmax(confusion_row - confusion_row[cell_idx])
        if confusion_row[misclass_idx] > 0 and misclass_idx != cell_idx:
            misclass_rate = (confusion_row[misclass_idx] / confusion_row.sum()) * 100
            confused_with = leaf_classes[misclass_idx]
            
            # Check if this is a hierarchy-related issue
            parent = ontology_dict.get(cell_type, None)
            confused_parent = ontology_dict.get(confused_with, None)
            is_related = (confused_parent == parent) or (parent == confused_with) or (confused_parent == cell_type)
            relation_note = " (HIERARCHY RELATED)" if is_related else ""
            
            print(f"{cell_type:<40} {confused_with:<40} {misclass_rate:<15.1f}%{relation_note}")

# Summary statistics
print(f"\n📈 PERFORMANCE DISTRIBUTION:")
all_f1 = per_class_metrics['f1'].values
print(f"   Mean F1: {all_f1.mean():.4f}")
print(f"   Median F1: {np.median(all_f1):.4f}")
print(f"   Std Dev: {all_f1.std():.4f}")
print(f"   Min F1: {all_f1.min():.4f} ({per_class_metrics[per_class_metrics['f1'] == all_f1.min()]['cell_type'].values[0]})")
print(f"   Max F1: {all_f1.max():.4f} ({per_class_metrics[per_class_metrics['f1'] == all_f1.max()]['cell_type'].values[0]})")
print(f"\n   Classes with F1 < 0.70: {(all_f1 < 0.70).sum()}")
print(f"   Classes with F1 < 0.80: {(all_f1 < 0.80).sum()}")
print(f"   Classes with F1 > 0.90: {(all_f1 > 0.90).sum()}")

# Visualize worst performers
fig, ax = plt.subplots(figsize=(12, 10))

worst_n = 20
worst_performers = per_class_metrics.tail(worst_n).sort_values('f1')
y_pos = np.arange(len(worst_performers))

colors = ['#d62728' if f1 < 0.70 else '#ff7f0e' if f1 < 0.80 else '#2ca02c' 
          for f1 in worst_performers['f1']]

ax.barh(y_pos, worst_performers['f1'].values, color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(worst_performers['cell_type'].values, fontsize=9)
ax.set_xlabel('F1 Score', fontsize=11)
ax.set_title(f'Worst {worst_n} Performing Cell Types (Red: F1<0.70, Orange: F1<0.80, Green: F1≥0.80)', 
             fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.set_xlim([0, 1.0])

# Add value labels
for i, v in enumerate(worst_performers['f1'].values):
    ax.text(v - 0.05, i, f'{v:.3f}', va='center', ha='right', fontsize=8)

plt.tight_layout()
worst_path = os.path.join(OUT_DIR, 'worst_performers.png')
plt.savefig(worst_path, dpi=200, bbox_inches='tight')
print(f"\n✅ Saved worst performers visualization to {worst_path}")
plt.show()

## 13. Key Improvements in This Fixed Version

**What Changed:**
1. ✅ **Hierarchy Building with Proper Stopping Logic** - Unknown/None values terminate traversal and the leaf column is always appended
2. ✅ **Corrected Reachability Matrix** - Parent→Child directionality with Floyd-Warshall transitive closure
3. ✅ **Expanded Logit Space** - Ancestors are included so HCE can grant credit to ancestor predictions
4. ✅ **Leaf Projection for Evaluation** - Ancestor predictions are projected down to leaves for metrics

**Expected Improvements vs. Previous Broken Version:**
- Previous: Only a handful of hierarchy relationships were active
- Now: All parent→child edges that exist in ontology are represented
- Previous: Ancestor predictions could not be credited
- Now: Ancestor credit is available during training while evaluation stays leaf-level